# Build panel features v2

This section merges approved national features and creates simple lag and change variables.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'

base = pd.read_csv(PROCESSED_DIR / 'panel_skeleton.csv')
feasibility = pd.read_csv(OUTPUTS_DIR / 'control_feasibility_extended.csv')
approved_new = feasibility.loc[feasibility['merge_approved'].eq('yes'), 'variable_name'].tolist()
panel = base.copy()
panel.shape, approved_new

((608, 9),
 ['physicians_per_100k',
  'hospital_beds_per_100k',
  'oop_health_expenditure_share_pc',
  'gini_income',
  'long_term_unemployment_rate_pc'])

In [2]:
# merge approved national features by country and year
for variable in approved_new:
    feature = pd.read_csv(PROCESSED_DIR / f'feature_{variable}.csv')
    panel = panel.merge(feature, on=['geo', 'year'], how='left')

panel = panel.sort_values(['geo', 'year']).reset_index(drop=True)
panel.head()

,geo,year,unmet_need_pc,status,gdp_per_capita_eur,unemployment_rate_pc,poverty_or_social_exclusion_pc,government_health_expenditure_gdp_pc,compulsory_health_financing_gdp_pc,physicians_per_100k,hospital_beds_per_100k,oop_health_expenditure_share_pc,gini_income,long_term_unemployment_rate_pc
0,AL,2017,13.1,NaN,4100.0,NaN,58.5,NaN,NaN,NaN,NaN,NaN,36.8,NaN
1,AL,2018,14.8,NaN,4540.0,NaN,53.9,NaN,NaN,NaN,NaN,NaN,35.4,NaN
2,AL,2019,14.6,NaN,4880.0,NaN,50.7,NaN,NaN,NaN,NaN,NaN,34.3,NaN
3,AL,2020,10.6,NaN,4710.0,NaN,46.2,NaN,NaN,NaN,NaN,NaN,33.2,NaN
4,AL,2021,10.7,NaN,5420.0,NaN,46.6,NaN,NaN,NaN,NaN,NaN,33.0,NaN


In [3]:
existing_controls = [
    'gdp_per_capita_eur',
    'unemployment_rate_pc',
    'poverty_or_social_exclusion_pc',
    'government_health_expenditure_gdp_pc',
    'compulsory_health_financing_gdp_pc',
]
panel['log_gdp_per_capita'] = np.log(panel['gdp_per_capita_eur'])
if 'oop_health_expenditure_share_pc' in panel.columns:
    panel['log_oop_health_expenditure_share'] = np.log1p(panel['oop_health_expenditure_share_pc'])

lag_sources = [
    'gdp_per_capita_eur',
    'unemployment_rate_pc',
    'poverty_or_social_exclusion_pc',
]
health_lag_candidates = [
    'physicians_per_100k',
    'hospital_beds_per_100k',
    'oop_health_expenditure_share_pc',
]
selected_health_feature = next((column for column in health_lag_candidates if column in panel.columns), None)
if selected_health_feature is not None:
    lag_sources.append(selected_health_feature)

for column in lag_sources:
    panel[f'{column}_lag1'] = panel.groupby('geo')[column].shift(1)

panel['gdp_per_capita_growth'] = (panel['gdp_per_capita_eur'] - panel['gdp_per_capita_eur_lag1']) / panel['gdp_per_capita_eur_lag1']
panel['unemployment_rate_change'] = panel['unemployment_rate_pc'] - panel['unemployment_rate_pc_lag1']
selected_health_feature

'physicians_per_100k'

In [4]:
feature_columns = existing_controls + approved_new + [
    'log_gdp_per_capita',
    'gdp_per_capita_eur_lag1',
    'unemployment_rate_pc_lag1',
    'poverty_or_social_exclusion_pc_lag1',
    'gdp_per_capita_growth',
    'unemployment_rate_change',
]
if selected_health_feature is not None:
    feature_columns.append(f'{selected_health_feature}_lag1')
if 'log_oop_health_expenditure_share' in panel.columns:
    feature_columns.append('log_oop_health_expenditure_share')

feature_columns = [column for column in feature_columns if column in panel.columns]
X_for_later_ml = panel[feature_columns].copy()
X_standardized = (X_for_later_ml - X_for_later_ml.mean()) / X_for_later_ml.std(ddof=0)
panel.to_csv(PROCESSED_DIR / 'panel_features_v2.csv', index=False)
pd.DataFrame({'feature': feature_columns, 'non_missing_rows': [int(panel[column].notna().sum()) for column in feature_columns]})

,feature,non_missing_rows
0,gdp_per_capita_eur,604
1,unemployment_rate_pc,556
2,poverty_or_social_exclusion_pc,375
3,government_health_expenditure_gdp_pc,502
4,compulsory_health_financing_gdp_pc,444
5,physicians_per_100k,431
6,hospital_beds_per_100k,522
7,oop_health_expenditure_share_pc,444
8,gini_income,409
9,long_term_unemployment_rate_pc,554


In [5]:
new_missing = panel[approved_new].isna().sum().sort_values(ascending=False) if approved_new else pd.Series(dtype='int64')
engineered = [column for column in feature_columns if column not in existing_controls and column not in approved_new]
summary = (
    '# Feature engineering summary\n\n'
    f'The extended panel keeps {len(panel)} observed country-year rows from the Phase 1 panel. '
    f'{len(approved_new)} new Eurostat feature datasets were merged.\n\n'
    f'The feature view contains {len(feature_columns)} columns: {len(existing_controls)} original controls, '
    f'{len(approved_new)} new variables, and {len(engineered)} transformed, lag, or change variables.\n\n'
    'Main missingness introduced by new features:\n'
)
if len(new_missing) == 0:
    summary += '- No approved new features were merged.\n'
else:
    for variable, count in new_missing.items():
        summary += f'- `{variable}` has {int(count)} missing rows after the left merge.\n'
summary += '\nLag variables are missing for the first observed year within each country. No values were carried forward.\n'
(OUTPUTS_DIR / 'feature_engineering_summary.md').write_text(summary, encoding='utf-8')
summary

'# Feature engineering summary\n\nThe extended panel keeps 608 observed country-year rows from the Phase 1 panel. 5 new Eurostat feature datasets were merged.\n\nThe feature view contains 18 columns: 5 original controls, 5 new variables, and 8 transformed, lag, or change variables.\n\nMain missingness introduced by new features:\n- `gini_income` has 199 missing rows after the left merge.\n- `physicians_per_100k` has 177 missing rows after the left merge.\n- `oop_health_expenditure_share_pc` has 164 missing rows after the left merge.\n- `hospital_beds_per_100k` has 86 missing rows after the left merge.\n- `long_term_unemployment_rate_pc` has 54 missing rows after the left merge.\n\nLag variables are missing for the first observed year within each country. No values were carried forward.\n'